# Select Transfer Function


In [1]:
import control as ctrl
import hickle as hkl
import sympy as sp

from lib.utils import (
    G_sp_to_ctrl,
    input_names,
    output_labels,
    output_names,
    output_units,
)

G_matrix = hkl.load("../outputs/G_simplified.hkl")
G_sym = sp.simplify(G_matrix[0, 2])  # G_{1,3}: T1/FR
G_name = f"{output_names[0]}_{input_names[2]}"
G_ylabel = f"{output_labels[0]} / {output_units[0]}"

s = sp.symbols("s")
t = sp.symbols("t", real=True, positive=True)

G = G_sp_to_ctrl(G_sym)


# Control


In [2]:
import numpy as np

t_num = np.linspace(0, 2, 1000)

_, y_control_impulse = ctrl.impulse_response(G, T=t_num)
_, y_control_step = ctrl.step_response(G, T=t_num)
_, y_control_ramp = ctrl.forced_response(G, T=t_num, U=t_num)


# Sympy


In [3]:
def symbolic_response(G_sym, U_sym, t_num):
    Y_sym = sp.simplify(G_sym * U_sym)

    y_t = sp.inverse_laplace_transform(Y_sym, s, t)
    y_t = sp.expand_complex(y_t).simplify()
    y_t = sp.re(y_t)
    y_t = sp.simplify(y_t)

    y_func = sp.lambdify(t, y_t, "numpy")
    y_analytic = np.real(y_func(t_num))

    return y_t, y_analytic


y_analytic_impulse, y_analytic_numeric_impulse = symbolic_response(G_sym, 1, t_num)
y_analytic_step, y_analytic_numeric_step = symbolic_response(G_sym, 1 / s, t_num)
y_analytic_ramp, y_analytic_numeric_ramp = symbolic_response(G_sym, 1 / (s**2), t_num)


# Plot


In [4]:
from lib.plots import plot_or_show, plt


def plot_response_comparison(t, y_control, y_analytic, response_name, save_path=None):
    fig, ax = plt.subplots(figsize=(8, 4))

    ax.plot(t, y_control, label=f"{response_name} (control)")
    ax.plot(t, y_analytic, "--", label=f"{response_name} (analítica)")

    ax.set_xlabel("Tempo / h")
    ax.set_ylabel(G_ylabel)
    ax.legend()

    plot_or_show(save_path)


plot_response_comparison(
    t_num,
    y_control_impulse,
    y_analytic_numeric_impulse,
    "Resposta ao impulso",
    f"responses/{G_name}_impulse",
)

plot_response_comparison(
    t_num,
    y_control_step,
    y_analytic_numeric_step,
    "Resposta ao degrau",
    f"responses/{G_name}_step",
)

plot_response_comparison(
    t_num,
    y_control_ramp,
    y_analytic_numeric_ramp,
    "Resposta à rampa",
    f"responses/{G_name}_ramp",
)


Plot saved to ../figures/responses/T1_FR_impulse.png
Plot saved to ../figures/responses/T1_FR_step.png
Plot saved to ../figures/responses/T1_FR_ramp.png


In [5]:
import re


def print_latex_response(expr, name):
    expr = sp.expand(expr)
    expr_round = expr.xreplace({n: round(float(n), 3) for n in expr.atoms(sp.Number)})
    expr = sp.simplify(expr)

    latex_str = sp.latex(expr_round)

    latex_str = sp.latex(expr_round)
    latex_str = re.sub(r"(?<=\d)\.(?=\d)", "{,}", latex_str)

    print(f"\n{name}")
    print("-" * len(name))
    print(latex_str)


print_latex_response(y_analytic_impulse, "Impulse Response")
print_latex_response(y_analytic_step, "Step Response")
print_latex_response(y_analytic_ramp, "Ramp Response")



Impulse Response
----------------
- 7{,}15 e^{- 48{,}038 t} \sin{\left(19{,}62 t \right)} - 1{,}074 e^{- 48{,}038 t} \cos{\left(19{,}62 t \right)} + 1{,}074 e^{- 3{,}179 t}

Step Response
-------------
0{,}267 + 0{,}12 e^{- 48{,}038 t} \sin{\left(19{,}62 t \right)} + 0{,}071 e^{- 48{,}038 t} \cos{\left(19{,}62 t \right)} - 0{,}338 e^{- 3{,}179 t}

Ramp Response
-------------
0{,}267 t - 0{,}104 - 0{,}002 e^{- 48{,}038 t} \sin{\left(19{,}62 t \right)} - 0{,}002 e^{- 48{,}038 t} \cos{\left(19{,}62 t \right)} + 0{,}106 e^{- 3{,}179 t}
